# 05 — Joint analysis: confidence × explanation trustworthiness

**Project:** Quantitatively Audited Explainable AI for Brain Tumor MRI Classification

Notebook 3 scored the explanations, image by image: faithfulness, stability, localization.
Notebook 4 scored the predictions: confidence, entropy, mutual information. This notebook joins
them and asks the question the project exists to answer:

> **When the model is confident, is its explanation also trustworthy? And when the explanation
> can't be trusted, does the model's uncertainty warn us?**

If yes, confidence works as a cheap gate: trust confident predictions *and* their heatmaps, and
send the rest for review. If no, the dangerous case is real: a **confident, wrong prediction with a
convincing-looking heatmap**, which no uncertainty threshold catches and which a heatmap may even
make *more* persuasive.

Nothing here loads the model or runs a forward pass. It analyses saved tables and renders saved
maps, so it runs on CPU.

**Inputs to attach** (Add Input → Your Work → Notebooks):

| input | used for | on `sys.path`? |
|---|---|---|
| notebook 3 output | `results/audit/audit_results.csv`, `audit_config.json` (lineage), `audit_summary.json` | no |
| notebook 4 output | `results/uncertainty/uncertainty_results.csv`, `uncertainty_summary.json` | no |
| notebook 2 output | `results/explanations/test_explanations.npz` + `meta.json` (maps for the panels) + `src/` | **yes, its `src/` only** (copied) |
| notebook 0 output | `manifest.csv` + `.mat` files (raw MRI for the panels) | no |

**Outputs** (`/kaggle/working/results/joint/`): `joint_results.csv` (merged, one row per test
image × explainer), `joint_summary.json` (every headline number with its patient-bootstrap CI, the
cross-tabs, the correlation table, the Grad-CAM-vs-IG verdict, lineage), `correlations.csv`, and
figures in `results/figures/`.

## 1. Setup

No GPU needed. If one is attached it's simply ignored, since nothing here runs the model.

In [ ]:
import hashlib
import json
import shutil
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage import measure

try:
    import torch
    print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available(), "(not needed)")
except ImportError:
    torch = None

WORK = Path("/kaggle/working")
JOINT_DIR = WORK / "results" / "joint"
FIG_DIR = WORK / "results" / "figures"
for d in [JOINT_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

## 2. Locate inputs and check lineage

Each table comes from a different commit, so the first job is to prove they describe **the same
model on the same split**. Where each one records its lineage:

| source | checkpoint MD5 | manifest MD5 |
|---|---|---|
| notebook 2 `meta.json` (the maps) | `checkpoint_md5` | `manifest_md5` |
| notebook 3 `audit_config.json` (the audit) | `checkpoint_md5` | `manifest_md5` |
| notebook 4 `uncertainty_summary.json` (uncertainty) | `lineage.checkpoint_md5` | `lineage.manifest_md5` |

Notebook 3's `audit_summary.json` doesn't record the MD5s; its sibling `audit_config.json` does,
which is why that file is used. If `audit_summary.json` ever does carry them, they're checked too.
All values must be identical; one mismatch stops the notebook. Tables from different models or
splits must not be joined, because every correlation below would then be meaningless.

The manifest used for rendering is the attached `manifest.csv` whose own MD5 matches.

In [ ]:
def find_unique(pattern, parent_names, what, how_to_fix):
    hits = [h for h in sorted(Path("/kaggle/input").rglob(pattern))
            if [p.name for p in h.parents[:len(parent_names)]] == parent_names]
    if len(hits) == 0:
        print("mounted under /kaggle/input:")
        for p in sorted(Path("/kaggle/input").glob("*/*"))[:40]:
            print("   ", p)
        raise FileNotFoundError(f"{what} not found under /kaggle/input. {how_to_fix}")
    if len(hits) > 1:
        raise RuntimeError(f"several copies of {what} attached, keep only one: {hits}")
    return hits[0]


AUDIT_CSV = find_unique("audit_results.csv", ["audit", "results"], "results/audit/audit_results.csv", "Attach notebook 3's committed output.")
UNC_CSV = find_unique("uncertainty_results.csv", ["uncertainty", "results"], "results/uncertainty/uncertainty_results.csv", "Attach notebook 4's committed output.")
NPZ_PATH = find_unique("test_explanations.npz", ["explanations", "results"], "results/explanations/test_explanations.npz", "Attach notebook 2's committed output.")
NB2_DIR = NPZ_PATH.parents[2]
for p in [AUDIT_CSV.parent / "audit_config.json", AUDIT_CSV.parent / "audit_summary.json", UNC_CSV.parent / "uncertainty_summary.json",
          NPZ_PATH.parent / "meta.json", NB2_DIR / "src/data/dataset.py", NB2_DIR / "src/data/transforms.py"]:
    if not p.exists():
        raise FileNotFoundError(f"missing {p} -- re-commit the notebook that produces it")

audit_cfg = json.loads((AUDIT_CSV.parent / "audit_config.json").read_text())
audit_sum = json.loads((AUDIT_CSV.parent / "audit_summary.json").read_text())
unc_sum = json.loads((UNC_CSV.parent / "uncertainty_summary.json").read_text())
expl_meta = json.loads((NPZ_PATH.parent / "meta.json").read_text())

lineage = {
    "notebook2_meta": (expl_meta["checkpoint_md5"], expl_meta["manifest_md5"]),
    "notebook3_audit_config": (audit_cfg["checkpoint_md5"], audit_cfg["manifest_md5"]),
    "notebook4_uncertainty_summary": (unc_sum["lineage"]["checkpoint_md5"], unc_sum["lineage"]["manifest_md5"]),
}
if "checkpoint_md5" in audit_sum:
    lineage["notebook3_audit_summary"] = (audit_sum["checkpoint_md5"], audit_sum.get("manifest_md5", audit_cfg["manifest_md5"]))
print(pd.DataFrame(lineage, index=["checkpoint_md5", "manifest_md5"]).T)
if len({v[0] for v in lineage.values()}) != 1 or len({v[1] for v in lineage.values()}) != 1:
    raise RuntimeError("LINEAGE MISMATCH: these outputs describe different checkpoints and/or splits. "
                       "Re-run the out-of-date notebook(s) on the same notebook-1 checkpoint before joining.")
CKPT_MD5, MANIFEST_MD5 = next(iter(lineage.values()))
print(f"\nlineage OK: checkpoint {CKPT_MD5}, manifest {MANIFEST_MD5}")

MANIFEST_PATH = DATA_ROOT = None
for m in sorted(Path("/kaggle/input").rglob("manifest.csv")):
    if hashlib.md5(m.read_bytes()).hexdigest() != MANIFEST_MD5:
        continue
    first = pd.read_csv(m, nrows=1).file_path.iloc[0]
    for root in [m.parent / "data", m.parent.parent / "data"]:
        if (root / first).exists():
            MANIFEST_PATH, DATA_ROOT = m, root
            break
    if MANIFEST_PATH:
        break
if MANIFEST_PATH is None:
    raise FileNotFoundError(f"no attached manifest.csv with md5 {MANIFEST_MD5} (+ data/). Attach notebook 0's output (for the exemplar MRIs).")
print("manifest:", MANIFEST_PATH)

### Code: notebook 2's `src/` only

Only one `src` package can be importable, so this notebook uses notebook 2's. It has the dataset
and transforms the exemplar panels need. Notebooks 3 and 4's `src/` trees are never put on the path.
The handful of statistics helpers this notebook needs (patient bootstrap, NaN-safe Spearman and
AUROC) are written to `src/joint/stats.py`. They follow the same methodology as notebooks 3 and 4,
and a few known-answer checks run straight after.

In [ ]:
shutil.copytree(NB2_DIR / "src", WORK / "src", dirs_exist_ok=True)
(WORK / "src" / "joint").mkdir(exist_ok=True)
(WORK / "src" / "joint" / "__init__.py").touch()
sys.path = [p for p in sys.path if not p.startswith("/kaggle/input")]
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
import src
assert Path(src.__file__).resolve().parent == (WORK / "src").resolve(), src.__file__
print("src package:", Path(src.__file__).parent)

In [ ]:
%%writefile /kaggle/working/src/joint/stats.py
import numpy as np
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score


def patient_bootstrap_indices(patient_ids, n_boot=1000, seed=42):
    # each draw: patients with replacement, every drawn patient brings all of its rows
    pids = np.asarray(patient_ids)
    groups = [np.flatnonzero(pids == p) for p in np.unique(pids)]
    rng = np.random.default_rng(seed)
    return [np.concatenate([groups[g] for g in rng.integers(0, len(groups), len(groups))]) for _ in range(n_boot)]


def bootstrap_ci(fn, arrays, boot_indices, alpha=0.05):
    vals = np.array([fn(*(a[idx] for a in arrays)) for idx in boot_indices], dtype=float)
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0:
        return [float("nan"), float("nan")]
    return [float(np.percentile(vals, 100 * alpha / 2)), float(np.percentile(vals, 100 * (1 - alpha / 2)))]


def excludes_zero(ci, null=0.0):
    return bool(np.isfinite(ci).all() and (ci[0] > null or ci[1] < null))


def spearman_nan(x, y, min_n=8):
    x, y = np.asarray(x, float), np.asarray(y, float)
    ok = np.isfinite(x) & np.isfinite(y)
    if ok.sum() < min_n or x[ok].std() == 0 or y[ok].std() == 0:
        return float("nan")
    return float(spearmanr(x[ok], y[ok])[0])


def auroc_nan(score, positive):
    s, p = np.asarray(score, float), np.asarray(positive, float)
    ok = np.isfinite(s) & np.isfinite(p)
    p = p[ok].astype(bool)
    if p.all() or not p.any():
        return float("nan")
    return float(roc_auc_score(p, s[ok]))


def mean_nan(x):
    x = np.asarray(x, float)
    return float(np.nanmean(x)) if np.isfinite(x).any() else float("nan")

In [ ]:
from src.joint import stats as S

assert S.spearman_nan(np.arange(20), np.arange(20) ** 2) == 1.0
assert np.isnan(S.spearman_nan(np.arange(5), np.arange(5)))                       # too few pairs
assert S.spearman_nan(np.r_[np.arange(20), np.nan], np.r_[-np.arange(20), 3]) == -1.0   # NaN pair dropped
assert S.auroc_nan([0.1, 0.2, 0.8, 0.9, np.nan], [0, 0, 1, 1, 1]) == 1.0
assert np.isnan(S.auroc_nan([0.1, 0.2], [0, 0]))
pid_test = np.array(list("aaabcc"))
assert all(((pid_test[i] == p).sum() % (pid_test == p).sum() == 0) for i in S.patient_bootstrap_indices(pid_test, 20) for p in set(pid_test[i]))
assert S.excludes_zero([0.1, 0.3]) and not S.excludes_zero([-0.1, 0.3])
print("stats helpers: known-answer checks passed")

## 3. Merge

- `audit_results.csv`: one row per image × method, where method ∈ {`gradcam`,
  `integrated_gradients`, `random`}. `random` is notebook 3's control map, not an explainer.
- `uncertainty_results.csv`: one row per image.

The joined table has one row per **image × explainer** (Grad-CAM, IG). Per-image fields
(uncertainty, labels, tumor size) are repeated on both of an image's rows, and the random
control's per-image scores are attached as `*_random` columns, for the faithfulness score below.

Checks before anything is analysed:
- every method has exactly `N` rows, in the same `file_path` order as notebook 4's table and
  notebook 2's `test_predictions.csv`
- labels and patient IDs agree across tables
- the prediction the explanations explain (notebook 3's `pred`, the fp32 deterministic prediction
  from notebook 2) agrees with notebook 4's `det_pred` on ≥ 99% of images
- the merge drops and duplicates no rows (`validate="many_to_one"`, plus a row count)

**Which confidence, which correctness?** An explanation explains one specific class: the
deterministic prediction. So correctness means *that* prediction is correct (`correct`), and
confidence is the **MC mean probability of that same class** (`mc_conf_explained`). For nearly
all images this equals `mc_mean_confidence`. Where the MC argmax differs from the explained class,
it's lower, which is right: the MC model is then less sure of the class being explained. The
number of such images is printed.

In [ ]:
METHODS = ["gradcam", "integrated_gradients"]
LABELS = {"gradcam": "Grad-CAM", "integrated_gradients": "Integrated Gradients"}
CLASS_NAMES = expl_meta["class_names"]

audit = pd.read_csv(AUDIT_CSV)
unc = pd.read_csv(UNC_CSV)
key = pd.read_csv(NPZ_PATH.parent / "test_predictions.csv")
N = len(unc)
assert unc.file_path.is_unique and len(key) == N
assert (unc.file_path == key.file_path).all(), "notebook 4 row order differs from test_predictions.csv"
assert set(audit.method) >= set(METHODS) | {"random"}, set(audit.method)
for m in METHODS + ["random"]:
    a = audit[audit.method == m].sort_values("row")
    assert len(a) == N and (a.file_path.to_numpy() == unc.file_path.to_numpy()).all(), f"audit rows for {m} don't line up"

a0 = audit[audit.method == "gradcam"].sort_values("row").reset_index(drop=True)
assert (a0.true.to_numpy() == unc.true.to_numpy()).all() and (a0.patient_id.to_numpy() == unc.patient_id.to_numpy()).all()
agree = (a0.pred.to_numpy() == unc.det_pred.to_numpy()).mean()
print(f"explained prediction (nb3) vs nb4 det_pred agreement: {agree:.4f}")
assert agree >= 0.99, "notebooks 3 and 4 disagree on the deterministic prediction -- same checkpoint?"

RAND_COLS = ["deletion_auc", "insertion_auc", "iou", "dice", "energy_ratio", "pointing_hit"]
rand = audit[audit.method == "random"].set_index("file_path")[RAND_COLS].add_suffix("_random")
expl = audit[audit.method.isin(METHODS)]
unc_cols = [c for c in unc.columns if c not in ("patient_id", "true", "true_name")]
joint = (expl.merge(unc[unc_cols], on="file_path", how="left", validate="many_to_one")
             .merge(rand, left_on="file_path", right_index=True, how="left", validate="many_to_one"))
assert len(joint) == len(expl) == len(METHODS) * N and joint.mc_pred.notna().all()
assert (joint.deletion_auc.notna() <= joint.deletion_auc_random.notna()).all(), "random-control scores missing for audited images"

p_mc = joint[[f"p_mc_{c}" for c in CLASS_NAMES]].to_numpy()
joint["mc_conf_explained"] = p_mc[np.arange(len(joint)), joint.pred.to_numpy()]
joint["mc_pred_is_explained"] = joint.mc_pred == joint.pred
print(f"images whose MC argmax differs from the explained class: {(~a0.pred.eq(unc.mc_pred)).sum()} of {N}")

## 4. Metric directions and the faithfulness score

Every correlation sign below depends on these, so here they are in one place:

| metric | from | **better / more** when |
|---|---|---|
| `deletion_auc` | nb 3 | **lower** = more faithful |
| `insertion_auc` | nb 3 | **higher** = more faithful |
| `faith_score` (defined below) | here | **higher** = more faithful; 0 = no better than a random map |
| `spearman_*`, `ssim_*`, `stab_*_mean` | nb 3 | **higher** = more stable |
| `iou`, `dice`, `pointing_hit`, `energy_ratio` | nb 3 | **higher** = better localized (`energy_ratio` 1 = chance) |
| `mc_conf_explained` | nb 4 | **higher** = more certain |
| `predictive_entropy`, `mutual_information` | nb 4 | **higher** = more *un*certain |

**Why raw deletion/insertion AUCs mislead here.** Both curves track the probability of the
predicted class, and they start or end at the image's own confidence. A model that's 99.9% sure
keeps a high probability under *any* pixel ordering, even random. So confident images get high
deletion AUCs (looks *less* faithful) and high insertion AUCs (looks *more* faithful) for reasons
that have nothing to do with the explanation. Correlating raw AUCs with confidence would partly
just measure confidence against itself.

The fix uses the random control notebook 3 already ran **on the same image**:

$$\text{faith\_score} = (\text{ins}_{m} - \text{ins}_{\text{random}}) - (\text{del}_{m} - \text{del}_{\text{random}})$$

This is "how much better than a random pixel ordering, on this image, does method *m* do at
predicting which pixels matter", with insertion and deletion combined. It cancels the image-level
offset, so confident images don't get a free pass. The quadrant analysis uses `faith_score`, and
the correlation table reports raw AUCs *and* `faith_score`, so you can see how much of a raw
correlation the confound accounts for.

Two more per-image summaries:
- `stab_spearman_mean` and `stab_ssim_mean`: the mean over notebook 3's three perturbations.
- `size_tertile`: rebuilt from the tertile edges notebook 4 stored in its summary. The CSV itself
  doesn't have that column.

Faithfulness and stability exist only for images in notebook 3's model-audit set
(`in_model_audit`). If notebook 3 ran with `subset_n`, the analyses that need them use fewer images,
and the `n` is reported each time.

In [ ]:
joint["faith_score"] = (joint.insertion_auc - joint.insertion_auc_random) - (joint.deletion_auc - joint.deletion_auc_random)
joint["faith_raw"] = joint.insertion_auc - joint.deletion_auc
SPEC_NAMES = [c[len("spearman_"):] for c in joint.columns if c.startswith("spearman_")]
joint["stab_spearman_mean"] = joint[[f"spearman_{n}" for n in SPEC_NAMES]].mean(axis=1, skipna=False)
joint["stab_ssim_mean"] = joint[[f"ssim_{n}" for n in SPEC_NAMES]].mean(axis=1, skipna=False)

edges = np.asarray(unc_sum["size_tertile_edges_tumor_frac"])
SIZE_NAMES = ["small", "medium", "large"]
joint["size_tertile"] = [SIZE_NAMES[b] for b in np.clip(np.searchsorted(edges, joint.tumor_frac, side="right") - 1, 0, 2)]

joint["method"] = pd.Categorical(joint.method, METHODS, ordered=True)
joint = joint.sort_values(["row", "method"]).reset_index(drop=True)
joint["method"] = joint.method.astype(str)
joint.to_csv(JOINT_DIR / "joint_results.csv", index=False)
n_faith = joint.groupby("method").faith_score.apply(lambda s: s.notna().sum()).to_dict()
print(f"joint_results.csv: {len(joint)} rows ({N} images x {len(METHODS)} explainers); faith_score available: {n_faith}")
joint.groupby("method")[["faith_score", "deletion_auc", "insertion_auc", "stab_spearman_mean", "energy_ratio", "mc_conf_explained"]].describe().T.round(3)

## 5. Bootstrap machinery

These are the same patient-cluster resamples as notebooks 3 and 4: the ~35 test patients drawn with
replacement, each bringing all of its slices. The **same** resample indices are used for every
statistic in this notebook, so all CIs come from identical draws. `report()` stores each value and
its CI in the summary. Per-method arrays `W[method][column]` are aligned to the image row order, so
one set of indices fits every method.

In [ ]:
SEED = unc_sum["settings"]["seed"]
N_BOOT = unc_sum["settings"].get("n_bootstrap", 1000)
pids = unc.patient_id.to_numpy()
BOOT = S.patient_bootstrap_indices(pids, n_boot=N_BOOT, seed=SEED)
SUMMARY = {}

W = {m: {c: joint.loc[joint.method == m].sort_values("row")[c].to_numpy(dtype=float)
         for c in joint.columns if pd.api.types.is_numeric_dtype(joint[c]) or pd.api.types.is_bool_dtype(joint[c])}
     for m in METHODS}
for m in METHODS:
    W[m]["wrong"] = 1.0 - W[m]["correct"]


def report(section, name, fn, *arrays):
    arrays = tuple(np.asarray(a, dtype=float) for a in arrays)
    value = float(fn(*arrays))
    ci = S.bootstrap_ci(fn, arrays, BOOT)
    node = SUMMARY
    for part in section.split("/"):
        node = node.setdefault(part, {})
    node[name] = {"value": value, "ci95_patient_bootstrap": ci, "ci_excludes_0": S.excludes_zero(ci)}
    return value, ci


def fmt(v, ci, signed=False):
    f = "{:+.3f}" if signed else "{:.3f}"
    return f"{f.format(v)} [{f.format(ci[0])}, {f.format(ci[1])}]"


print(f"{N_BOOT} patient-bootstrap resamples over {len(np.unique(pids))} patients")

## 6. Analysis 1 — correlations between certainty and explanation trust

Spearman correlation of each certainty measure with each trust metric, per explainer, each with a
patient-bootstrap CI. **A correlation counts as real only if its CI excludes 0.**

To make the heatmap readable at a glance, it shows **oriented** correlations. Both sides are first
flipped to point the "good" way: certainty = `mc_conf_explained`, `−predictive_entropy`,
`−mutual_information`; trust = each metric × its direction, so `−deletion_auc`. Then, everywhere:

- **positive (blue)** = more certain predictions come with more trustworthy explanations
- **negative (red)** = more certain predictions come with *less* trustworthy explanations
- `*` = CI excludes 0

The raw (un-oriented) correlations and CIs are in `correlations.csv`. Compare the `deletion_auc` /
`insertion_auc` columns with `faith_score`: the gap shows how much of a raw faithfulness–confidence
correlation was the mechanical confound from §4.

In [ ]:
CERTAINTY = {"mc_conf_explained": +1, "predictive_entropy": -1, "mutual_information": -1}
TRUST = {"faith_score": +1, "deletion_auc": -1, "insertion_auc": +1,
         "stab_spearman_mean": +1, "stab_ssim_mean": +1,
         **{f"spearman_{n}": +1 for n in SPEC_NAMES}, **{f"ssim_{n}": +1 for n in SPEC_NAMES},
         "iou": +1, "dice": +1, "pointing_hit": +1, "energy_ratio": +1}

t0 = time.time()
rows = []
for m in METHODS:
    for u, us in CERTAINTY.items():
        for t, ts in TRUST.items():
            v, ci = report(f"correlations/{m}", f"{u}~{t}", S.spearman_nan, W[m][u], W[m][t])
            sign = us * ts
            oci = sorted([sign * ci[0], sign * ci[1]])
            rows.append({"method": m, "certainty": u, "trust_metric": t, "spearman": v, "ci_lo": ci[0], "ci_hi": ci[1],
                         "oriented": sign * v, "oriented_ci_lo": oci[0], "oriented_ci_hi": oci[1],
                         "real": S.excludes_zero(ci), "n": int((np.isfinite(W[m][u]) & np.isfinite(W[m][t])).sum())})
corr = pd.DataFrame(rows)
corr.to_csv(JOINT_DIR / "correlations.csv", index=False)
print(f"{len(corr)} correlations x {N_BOOT} resamples in {time.time() - t0:.0f}s")

show = corr[corr.trust_metric.isin(["faith_score", "deletion_auc", "insertion_auc", "stab_spearman_mean", "energy_ratio", "pointing_hit"])]
display(show.assign(oriented_ci=lambda d: [fmt(o, [a, b], True) for o, a, b in zip(d.oriented, d.oriented_ci_lo, d.oriented_ci_hi)])
        .pivot_table(index=["method", "certainty"], columns="trust_metric", values="oriented_ci", aggfunc="first"))

In [ ]:
cols = list(TRUST)
fig, axes = plt.subplots(len(METHODS), 1, figsize=(15, 2.2 + 1.3 * len(CERTAINTY) * len(METHODS) / 2), squeeze=False)
vmax = max(0.3, np.nanmax(np.abs(corr.oriented)))
for ax, m in zip(axes[:, 0], METHODS):
    sub = corr[corr.method == m]
    mat = sub.pivot(index="certainty", columns="trust_metric", values="oriented").reindex(index=list(CERTAINTY), columns=cols)
    real = sub.pivot(index="certainty", columns="trust_metric", values="real").reindex(index=list(CERTAINTY), columns=cols)
    im = ax.imshow(mat.to_numpy(float), cmap="RdBu", vmin=-vmax, vmax=vmax, aspect="auto")
    for r_ in range(mat.shape[0]):
        for c_ in range(mat.shape[1]):
            v = mat.iat[r_, c_]
            if np.isfinite(v):
                ax.text(c_, r_, f"{v:+.2f}{'*' if real.iat[r_, c_] else ''}", ha="center", va="center", fontsize=7,
                        color="white" if abs(v) > 0.6 * vmax else "black")
    ax.set_yticks(range(len(CERTAINTY)), ["confidence", "-entropy", "-mutual info"])
    ax.set_xticks(range(len(cols)), cols, rotation=45, ha="right", fontsize=8)
    ax.set_title(f"{LABELS[m]}: oriented Spearman (blue = more certain <-> more trustworthy; * = patient-bootstrap CI excludes 0)", fontsize=10)
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.8, label="oriented Spearman")
fig.savefig(FIG_DIR / "joint_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Analysis 2 — the central cross-tab: confident/uncertain × faithful/unfaithful

Each image (per explainer) is put in one of four cells. The key quantity is the **accuracy within
each cell**, because it answers whether a good-looking combination actually means a correct
prediction.

| | **faithful** | **unfaithful** |
|---|---|---|
| **confident** | ideal: trust the prediction *and* the heatmap | **danger**: a confident prediction whose heatmap doesn't reflect what drove it |
| **uncertain** | the model knows it's unsure; the heatmap is honest about why | send for review anyway |

The **danger** count is reported on its own, together with the subset of it that is also **wrong**:
*confidently wrong with an untrustworthy explanation*, the failure mode that matters most.

Two definitions, so the result doesn't hinge on one threshold:
1. **Median split** (primary): confident = `mc_conf_explained` ≥ median, faithful = `faith_score`
   ≥ median of that explainer. Balanced cells, the most statistical power. But "uncertain" here
   only means *less* confident than the typical image, which may still be 99%.
2. **Absolute** (sensitivity check): confident = `mc_conf_explained` ≥ 0.9; faithful =
   `faith_score` > 0, i.e. better than a random map on this image. More meaningful cut-offs, but
   cells can be small or empty.

In [ ]:
def crosstab(m, confident, faithful, label):
    w = W[m]
    ok = np.isfinite(w["faith_score"])
    out = {}
    for cell, cmask in [("confident_faithful", confident & faithful), ("confident_unfaithful", confident & ~faithful),
                        ("uncertain_faithful", ~confident & faithful), ("uncertain_unfaithful", ~confident & ~faithful)]:
        mk = (cmask & ok).astype(float)
        n = int(mk.sum())
        n_wrong = int(((mk > 0) & (w["wrong"] > 0)).sum())
        v, ci = report(f"crosstab/{label}/{m}/accuracy", cell,
                       lambda c, k: c[k > 0].mean() if (k > 0).any() else np.nan, w["correct"], mk)
        out[cell] = {"n": n, "n_wrong": n_wrong, "accuracy": v, "accuracy_ci": ci,
                     "mean_entropy": float(np.nanmean(w["predictive_entropy"][mk > 0])) if n else float("nan")}
    danger = confident & ~faithful & ok
    out["danger_confident_unfaithful_wrong"] = int((danger & (w["wrong"] > 0)).sum())
    out["n_errors_total_in_audit"] = int(((w["wrong"] > 0) & ok).sum())
    out["share_of_errors_that_are_confident_unfaithful"] = (out["danger_confident_unfaithful_wrong"] / out["n_errors_total_in_audit"]
                                                            if out["n_errors_total_in_audit"] else float("nan"))
    SUMMARY.setdefault("crosstab", {}).setdefault(label, {}).setdefault(m, {}).update(
        {k: v for k, v in out.items() if not isinstance(v, dict)}, cells={k: v for k, v in out.items() if isinstance(v, dict)})
    return out


def show_crosstab(out, m, label):
    tab = pd.DataFrame({k: out[k] for k in ["confident_faithful", "confident_unfaithful", "uncertain_faithful", "uncertain_unfaithful"]}).T
    tab["accuracy [95% CI]"] = [fmt(r["accuracy"], r["accuracy_ci"]) if r["n"] else "-" for r in tab.to_dict("records")]
    print(f"\n{LABELS[m]} | {label}: danger (confident + unfaithful + WRONG) = {out['danger_confident_unfaithful_wrong']} "
          f"of {out['n_errors_total_in_audit']} audited errors")
    display(tab[["n", "n_wrong", "accuracy [95% CI]", "mean_entropy"]])


DEFS = {}
for m in METHODS:
    conf, faith = W[m]["mc_conf_explained"], W[m]["faith_score"]
    ok = np.isfinite(faith)
    DEFS[("median", m)] = (conf >= np.median(conf[ok]), faith >= np.nanmedian(faith))
    DEFS[("absolute", m)] = (conf >= 0.9, faith > 0)
    SUMMARY.setdefault("crosstab_thresholds", {})[m] = {"median_confidence": float(np.median(conf[ok])),
                                                        "median_faith_score": float(np.nanmedian(faith)),
                                                        "absolute_confidence": 0.9, "absolute_faith_score": 0.0}
for label in ["median", "absolute"]:
    for m in METHODS:
        c_, f_ = DEFS[(label, m)]
        show_crosstab(crosstab(m, c_, f_, label), m, label)

The quadrant scatter shows the same thing as a picture. Confidence is plotted as **1 − confidence
on a log axis**, because nearly all predictions sit between 0.99 and 1.0 and a linear axis would
squash them into a single line. Right means *less* confident, up means *more* faithful. Wrong
predictions are red. The danger quadrant is **lower-left**: confident (left) and unfaithful (low).

In [ ]:
fig, axes = plt.subplots(1, len(METHODS), figsize=(13, 5), sharey=True)
for ax, m in zip(axes, METHODS):
    w = W[m]
    ok = np.isfinite(w["faith_score"])
    x = np.clip(1 - w["mc_conf_explained"], 1e-6, 1)
    right = ok & (w["wrong"] == 0)
    bad = ok & (w["wrong"] > 0)
    ax.scatter(x[right], w["faith_score"][right], s=10, alpha=0.45, color="#555", label=f"correct (n={right.sum()})")
    ax.scatter(x[bad], w["faith_score"][bad], s=45, marker="x", color="crimson", label=f"wrong (n={bad.sum()})")
    th = SUMMARY["crosstab_thresholds"][m]
    ax.axvline(1 - th["median_confidence"], color="k", lw=0.8, ls="--")
    ax.axhline(th["median_faith_score"], color="k", lw=0.8, ls="--")
    ax.axhline(0, color="#999", lw=0.8, ls=":")
    ax.set_xscale("log"); ax.invert_xaxis()
    ax.set_xlabel("1 - MC confidence of explained class (log; left = more confident)")
    ax.set_title(f"{LABELS[m]}\ndanger quadrant = lower-left; dotted = random-map level", fontsize=10)
    ax.legend(fontsize=8, frameon=False)
axes[0].set_ylabel("faith_score (higher = more faithful)")
plt.tight_layout()
fig.savefig(FIG_DIR / "joint_quadrant_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Analysis 3 — does uncertainty flag untrustworthy explanations?

The correlations in §6 say whether certainty and trust *move together*. The operational question is
sharper: **if I flag the most uncertain predictions, do I catch the bad explanations?** For each
uncertainty score, the AUROC for detecting:

- **unfaithful**: `faith_score` below that explainer's median
- **worse than random**: `faith_score` < 0
- **pointing miss**: the explanation's peak is outside the tumor
- **off-tumor energy**: `energy_ratio` ≤ 1, no more of the map's mass on the tumor than chance

0.5 = uncertainty carries no information about explanation quality. A flag counts as **usable**
only if the CI's lower bound is above 0.5. If none is, that's a central finding in its own right:
the model's uncertainty can't be used to decide when to distrust its heatmap.

In [ ]:
UNC_SCORES = {"predictive_entropy": lambda w: w["predictive_entropy"], "mutual_information": lambda w: w["mutual_information"],
              "1-confidence": lambda w: 1 - w["mc_conf_explained"]}
flag_rows = []
for m in METHODS:
    w = W[m]
    fs, er = w["faith_score"], w["energy_ratio"]
    targets = {
        "unfaithful(<median)": np.where(np.isfinite(fs), (fs < np.nanmedian(fs)).astype(float), np.nan),
        "worse_than_random(<0)": np.where(np.isfinite(fs), (fs < 0).astype(float), np.nan),
        "pointing_miss": np.where(np.isfinite(w["pointing_hit"]), 1 - w["pointing_hit"], np.nan),
        "off_tumor_energy(<=1)": np.where(np.isfinite(er), (er <= 1).astype(float), np.nan),
    }
    for tname, tgt in targets.items():
        for sname, sfn in UNC_SCORES.items():
            v, ci = report(f"uncertainty_flags/{m}/{tname}", sname, S.auroc_nan, sfn(w), tgt)
            flag_rows.append({"method": m, "target": tname, "score": sname, "auroc": v, "ci": fmt(v, ci),
                              "usable(ci_lo>0.5)": bool(np.isfinite(ci[0]) and ci[0] > 0.5),
                              "n_positive": int(np.nansum(tgt)), "n": int(np.isfinite(tgt).sum())})
flags = pd.DataFrame(flag_rows)
display(flags.pivot_table(index=["method", "target"], columns="score", values="ci", aggfunc="first"))
usable = flags[flags["usable(ci_lo>0.5)"]]
SUMMARY["uncertainty_flags_usable"] = usable[["method", "target", "score"]].to_dict("records")
print("usable flags:", "none" if usable.empty else usable[["method", "target", "score", "ci"]].to_string(index=False))

## 9. Analysis 4 — correct vs incorrect predictions

Do the errors come with worse explanations? With higher uncertainty? Means for correct and wrong
predictions, and the difference **wrong − correct** with a patient-bootstrap CI. With only ~20
errors, expect wide CIs. A difference that holds up despite that is a strong result.

In [ ]:
CMP = ["faith_score", "deletion_auc", "insertion_auc", "stab_spearman_mean", "stab_ssim_mean", "energy_ratio", "iou",
       "pointing_hit", "mc_conf_explained", "predictive_entropy", "mutual_information"]
cmp_rows = []
for m in METHODS:
    w = W[m]
    for c in CMP:
        vc, cic = report(f"correct_vs_wrong/{m}/{c}", "correct", lambda v, wr: S.mean_nan(v[wr == 0]), w[c], w["wrong"])
        vw, ciw = report(f"correct_vs_wrong/{m}/{c}", "wrong", lambda v, wr: S.mean_nan(v[wr > 0]) if (wr > 0).any() else np.nan, w[c], w["wrong"])
        vd, cid = report(f"correct_vs_wrong/{m}/{c}", "wrong_minus_correct",
                         lambda v, wr: (S.mean_nan(v[wr > 0]) - S.mean_nan(v[wr == 0])) if (wr > 0).any() else np.nan, w[c], w["wrong"])
        cmp_rows.append({"method": m, "metric": c, "correct": fmt(vc, cic), "wrong": fmt(vw, ciw),
                         "wrong - correct": fmt(vd, cid, True), "real": S.excludes_zero(cid)})
display(pd.DataFrame(cmp_rows).set_index(["method", "metric"]))

show_m = ["faith_score", "stab_spearman_mean", "energy_ratio", "predictive_entropy"]
fig, axes = plt.subplots(len(METHODS), len(show_m), figsize=(16, 7.5))
for r_, m in enumerate(METHODS):
    for c_, col in enumerate(show_m):
        ax = axes[r_, c_]
        v = W[m][col]
        data = [v[(W[m]["wrong"] == 0) & np.isfinite(v)], v[(W[m]["wrong"] > 0) & np.isfinite(v)]]
        bp = ax.boxplot(data, patch_artist=True, widths=0.55, showfliers=False)
        for patch, colr in zip(bp["boxes"], ["#888", "crimson"]):
            patch.set_facecolor(colr); patch.set_alpha(0.45)
        for k, (vals, colr) in enumerate(zip(data, ["#555", "crimson"]), start=1):
            ax.scatter(k + np.random.default_rng(k).uniform(-0.15, 0.15, len(vals)), vals, s=5 if k == 1 else 20, color=colr, alpha=0.5)
        ax.set_xticks([1, 2], [f"correct ({len(data[0])})", f"wrong ({len(data[1])})"], fontsize=8)
        d_ = SUMMARY["correct_vs_wrong"][m][col]["wrong_minus_correct"]
        ax.set_title(f"{LABELS[m]} | {col}\nwrong-correct {fmt(d_['value'], d_['ci95_patient_bootstrap'], True)}", fontsize=8)
        if col == "predictive_entropy":
            ax.set_yscale("symlog", linthresh=1e-3)
plt.tight_layout()
fig.savefig(FIG_DIR / "joint_correct_vs_wrong.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Analysis 5 — Grad-CAM vs IG

Two questions:

1. **Which explainer is more trustworthy on each axis?** The paired per-image difference, oriented
   so that **positive = Grad-CAM better**, with a patient-bootstrap CI. The verdict for an axis
   names a winner only if the CI excludes 0; otherwise it's a tie *on this test set*.
   - Read the SSIM row with notebook 3's sparse-map caveat in mind: shared empty background
     inflates Grad-CAM's SSIM.
2. **Do they agree on *which* images are well or poorly explained?** Spearman across images between
   the two methods' `faith_score` (and `energy_ratio`), plus the overlap of their worst-explained
   quartiles. Low agreement is itself a trust problem: "is this image's explanation trustworthy?"
   would then depend on which explainer was picked.

In [ ]:
AXES = {"faithfulness": ("faith_score", +1), "stability (Spearman)": ("stab_spearman_mean", +1),
        "stability (SSIM, sparse-map caveat)": ("stab_ssim_mean", +1), "localization (energy ratio)": ("energy_ratio", +1),
        "localization (IoU)": ("iou", +1), "localization (pointing)": ("pointing_hit", +1)}
gc, ig = W["gradcam"], W["integrated_gradients"]
verdict_rows = []
for axis, (col, sgn) in AXES.items():
    v, ci = report("gradcam_vs_ig/paired_difference_gradcam_better", col,
                   lambda a, b: S.mean_nan(sgn * (a - b)), gc[col], ig[col])
    winner = "Grad-CAM" if ci[0] > 0 else "Integrated Gradients" if ci[1] < 0 else "no difference distinguishable"
    SUMMARY["gradcam_vs_ig"].setdefault("verdict", {})[axis] = winner
    verdict_rows.append({"axis": axis, "metric": col, "GC - IG (oriented)": fmt(v, ci, True), "verdict": winner})
display(pd.DataFrame(verdict_rows).set_index("axis"))

agree_rows = []
for col in ["faith_score", "energy_ratio", "stab_spearman_mean"]:
    v, ci = report("gradcam_vs_ig/per_image_agreement_spearman", col, S.spearman_nan, gc[col], ig[col])
    agree_rows.append({"metric": col, "Spearman across images": fmt(v, ci)})
ok = np.isfinite(gc["faith_score"]) & np.isfinite(ig["faith_score"])
q_gc = set(np.flatnonzero(ok & (gc["faith_score"] <= np.nanpercentile(gc["faith_score"], 25))))
q_ig = set(np.flatnonzero(ok & (ig["faith_score"] <= np.nanpercentile(ig["faith_score"], 25))))
jac = len(q_gc & q_ig) / max(1, len(q_gc | q_ig))
SUMMARY["gradcam_vs_ig"]["worst_quartile_faith_jaccard"] = jac
display(pd.DataFrame(agree_rows).set_index("metric"))
print(f"overlap of the two methods' worst-explained quartiles (faith_score), Jaccard = {jac:.2f} "
      f"(about 0.14 expected if unrelated, 1 = identical sets)")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
for ax, col in zip(axes, ["faith_score", "energy_ratio"]):
    okk = np.isfinite(gc[col]) & np.isfinite(ig[col])
    wr = gc["wrong"] > 0
    ax.scatter(gc[col][okk & ~wr], ig[col][okk & ~wr], s=9, alpha=0.45, color="#555", label="correct")
    ax.scatter(gc[col][okk & wr], ig[col][okk & wr], s=40, marker="x", color="crimson", label="wrong")
    lo, hi = np.nanmin([gc[col][okk].min(), ig[col][okk].min()]), np.nanmax([gc[col][okk].max(), ig[col][okk].max()])
    ax.plot([lo, hi], [lo, hi], "k--", lw=0.8)
    a = SUMMARY["gradcam_vs_ig"]["per_image_agreement_spearman"][col]
    ax.set_title(f"{col}: per-image agreement, Spearman {fmt(a['value'], a['ci95_patient_bootstrap'])}", fontsize=9)
    ax.set_xlabel("Grad-CAM"); ax.set_ylabel("Integrated Gradients")
axes[0].legend(fontsize=8, frameon=False)
plt.tight_layout()
fig.savefig(FIG_DIR / "joint_gradcam_vs_ig_agreement.png", dpi=150, bbox_inches="tight")
plt.show()

## 11. Analysis 6 — by class and by tumor size

The same quantities split by true class and by tumor-size tertile (edges from notebook 4). Class and
size are confounded, since pituitary tumors are small, so read the two breakdowns together. If, say,
faithfulness collapses only for small tumors, that reflects the 7×7 Grad-CAM resolution or IG's
speckle on tiny targets (notebook 3), not a property of one class.

In [ ]:
BREAK = ["faith_score", "stab_spearman_mean", "energy_ratio", "iou", "mc_conf_explained", "predictive_entropy", "correct"]
true_arr = W["gradcam"]["true"]
size_arr = np.array([SIZE_NAMES.index(s) for s in joint.loc[joint.method == "gradcam"].sort_values("row").size_tertile])
brows = []
for gname, garr, names in [("class", true_arr, CLASS_NAMES), ("size", size_arr, SIZE_NAMES)]:
    for k, lvl in enumerate(names):
        g = (garr == k).astype(float)
        for m in METHODS:
            row = {"group": f"{gname}:{lvl}", "method": m, "n": int(g.sum())}
            for c in BREAK:
                v, ci = report(f"breakdown/{gname}/{lvl}/{m}", c, lambda x, gg: S.mean_nan(x[gg > 0]) if (gg > 0).any() else np.nan, W[m][c], g)
                row[c] = fmt(v, ci)
                row[f"_{c}_v"], row[f"_{c}_ci"] = v, ci
            brows.append(row)
brk = pd.DataFrame(brows)
display(brk[["group", "method", "n", *BREAK]].rename(columns={"correct": "accuracy"}).set_index(["group", "method"]))

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for r_, (gname, names) in enumerate([("class", CLASS_NAMES), ("size", SIZE_NAMES)]):
    for c_, col in enumerate(["faith_score", "energy_ratio"]):
        ax = axes[r_, c_]
        xpos = np.arange(len(names))
        for j, (m, colr) in enumerate(zip(METHODS, ["#d95f02", "#1b9e77"])):
            sub = brk[(brk.method == m) & brk.group.str.startswith(gname + ":")]
            vals = sub[f"_{col}_v"].to_numpy()
            cis = np.array(sub[f"_{col}_ci"].tolist())
            ax.bar(xpos + (j - 0.5) * 0.38, vals, width=0.36, color=colr, alpha=0.8, label=LABELS[m],
                   yerr=[vals - cis[:, 0], cis[:, 1] - vals], capsize=3)
        ax.set_xticks(xpos, names)
        ax.axhline(0 if col == "faith_score" else 1, color="k", lw=0.8, ls="--")
        ax.set_title(f"{col} by {gname} (bars: patient-bootstrap 95% CI; dashed = random/chance level)", fontsize=9)
axes[0, 0].legend(fontsize=8, frameon=False)
plt.tight_layout()
fig.savefig(FIG_DIR / "joint_breakdown_class_size.png", dpi=150, bbox_inches="tight")
plt.show()

## 12. Qualitative exemplars

The numbers above say *how often*; these panels show *what it looks like*. Images are drawn
deterministically (seed 42) from five categories, using the **median-split** definitions from §7.
Since the panels show both explainers, an image's "faithful" status here combines the two: the mean
of its within-method percentile ranks of `faith_score` ≥ 0.5.

1. **confident · faithful · correct**: the ideal case
2. **confident · unfaithful · WRONG**: the danger case
3. **confident · unfaithful · correct**: right, but the heatmap doesn't show why
4. **uncertain · WRONG**: the model hesitated, and was right to
5. **uncertain · correct**

Up to 2 per category. An empty category is reported as such. An empty danger category is good news,
and the write-up should say so.

Each row: raw MRI | Grad-CAM overlay | IG overlay | ground-truth mask contour. The maps come
straight from notebook 2's saved `.npz`; nothing is recomputed. The panels show class, MC
confidence and entropy, plus each method's `faith_score`, energy ratio, IoU and mean stability.

In [ ]:
from src.data.dataset import BrainTumorDataset
from src.data.transforms import get_eval_transforms, IMAGENET_MEAN, IMAGENET_STD

z = np.load(NPZ_PATH)
assert (z["file_path"] == unc.file_path.to_numpy()).all(), "npz row order differs"
manifest = pd.read_csv(MANIFEST_PATH)
test_ds = BrainTumorDataset(manifest, DATA_ROOT, "test", get_eval_transforms(expl_meta["image_size"]))
assert list(test_ds.df.file_path) == list(unc.file_path)

pct = {m: pd.Series(W[m]["faith_score"]).rank(pct=True).to_numpy() for m in METHODS}
joint_pct = np.nanmean(np.vstack([pct[m] for m in METHODS]), axis=0)
conf = W["gradcam"]["mc_conf_explained"]
ok_f = np.isfinite(joint_pct)
confident = conf >= np.median(conf[ok_f])
faithful = joint_pct >= 0.5
correct = W["gradcam"]["correct"] > 0
CATS = {
    "confident · faithful · correct": ok_f & confident & faithful & correct,
    "confident · unfaithful · WRONG": ok_f & confident & ~faithful & ~correct,
    "confident · unfaithful · correct": ok_f & confident & ~faithful & correct,
    "uncertain · WRONG": ok_f & ~confident & ~correct,
    "uncertain · correct": ok_f & ~confident & correct,
}
rng = np.random.default_rng(42)
exemplars = []
for cat, mask_ in CATS.items():
    pool = np.flatnonzero(mask_)
    picks = np.sort(rng.choice(pool, size=min(2, len(pool)), replace=False)) if len(pool) else []
    print(f"{cat:34s} pool {len(pool):4d} -> {list(map(int, picks))}")
    exemplars += [(cat, int(i)) for i in picks]
SUMMARY["exemplars"] = [{"category": c, "row": i, "file_path": unc.file_path[i]} for c, i in exemplars]

In [ ]:
def to_display(t):
    return (t[0].numpy() * IMAGENET_STD[0] + IMAGENET_MEAN[0]).clip(0, 1)


def contour(ax, mask, **kw):
    for c in measure.find_contours(mask.astype(float), 0.5):
        ax.plot(c[:, 1], c[:, 0], **kw)


def trust_line(m, i):
    w = W[m]
    f = lambda v: "n/a" if not np.isfinite(v) else f"{v:.2f}"
    return f"faith {f(w['faith_score'][i])} | energy x{f(w['energy_ratio'][i])} | IoU {f(w['iou'][i])} | stab {f(w['stab_spearman_mean'][i])}"


if not exemplars:
    print("no exemplars selected")
else:
    fig, axes = plt.subplots(len(exemplars), 4, figsize=(15, 3.6 * len(exemplars)))
    axes = np.atleast_2d(axes)
    for r_, (cat, i) in enumerate(exemplars):
        img, msk = to_display(test_ds[i]["image"]), z["mask"][i]
        t, p = int(unc.true[i]), int(W["gradcam"]["pred"][i])
        wrong = t != p
        axes[r_, 0].imshow(img, cmap="gray")
        axes[r_, 0].set_title(f"[{cat}]\ntrue {CLASS_NAMES[t]} | pred {CLASS_NAMES[p]}\n"
                              f"MC conf {conf[i]:.3f} | entropy {unc.predictive_entropy[i]:.3f}",
                              fontsize=8, color="crimson" if wrong else "black")
        for c_, (m, key_) in enumerate([("gradcam", "gradcam"), ("integrated_gradients", "ig")], start=1):
            axes[r_, c_].imshow(img, cmap="gray")
            axes[r_, c_].imshow(z[key_][i].astype(np.float32), cmap="jet", alpha=0.45, vmin=0, vmax=1)
            contour(axes[r_, c_], msk, color="white", lw=0.8, ls="--")
            axes[r_, c_].set_title(f"{LABELS[m]}\n{trust_line(m, i)}", fontsize=7.5)
        axes[r_, 3].imshow(img, cmap="gray")
        contour(axes[r_, 3], msk, color="red", lw=1.3)
        axes[r_, 3].set_title(f"tumor mask ({100 * unc.tumor_frac[i]:.2f}% of image)\n{Path(unc.file_path[i]).name}", fontsize=8)
    for ax in axes.ravel():
        ax.axis("off")
    plt.tight_layout()
    fig.savefig(FIG_DIR / "joint_exemplars.png", dpi=130, bbox_inches="tight")
    plt.show()

## 13. Verdict, summary, reload check

The verdict below is **generated from the CIs**, not written in advance. Each sentence takes one
branch or the other depending on whether the relevant interval excludes 0 (or 0.5 for AUROCs).
That keeps the stated conclusion tied to the evidence. Rewrite it in your own words for the
write-up, but don't make claims stronger than these sentences.

In [ ]:
def corr_entry(m, u, t):
    return SUMMARY["correlations"][m][f"{u}~{t}"]


verdict = []
for m in METHODS:
    e = corr_entry(m, "mc_conf_explained", "faith_score")
    v, ci = e["value"], e["ci95_patient_bootstrap"]
    if e["ci_excludes_0"] and v > 0:
        verdict.append(f"{LABELS[m]}: more confident predictions come with MORE faithful explanations (Spearman {fmt(v, ci)}); "
                       "confidence is a weak-to-useful proxy for explanation faithfulness.")
    elif e["ci_excludes_0"] and v < 0:
        verdict.append(f"{LABELS[m]}: more confident predictions come with LESS faithful explanations (Spearman {fmt(v, ci)}); "
                       "confidence is a misleading proxy for explanation faithfulness.")
    else:
        verdict.append(f"{LABELS[m]}: no detectable relation between confidence and faithfulness (Spearman {fmt(v, ci)}); "
                       "confidence is NOT a reliable proxy for explanation quality.")
    us = [r for r in SUMMARY["uncertainty_flags_usable"] if r["method"] == m]
    verdict.append(f"{LABELS[m]}: uncertainty " + ("CAN flag " + ", ".join(sorted({r['target'] for r in us})) if us else
                   "does NOT usefully flag untrustworthy explanations (no AUROC CI above 0.5)") + ".")
    ct = SUMMARY["crosstab"]["median"][m]
    verdict.append(f"{LABELS[m]}: {ct['danger_confident_unfaithful_wrong']} of {ct['n_errors_total_in_audit']} audited errors are "
                   "confidently wrong with an unfaithful explanation (median split).")
verdict.append("Grad-CAM vs IG: " + "; ".join(f"{a}: {w}" for a, w in SUMMARY["gradcam_vs_ig"]["verdict"].items()) + ".")
a = SUMMARY["gradcam_vs_ig"]["per_image_agreement_spearman"]["faith_score"]
verdict.append(f"The two explainers {'agree' if a['ci_excludes_0'] and a['value'] > 0 else 'do not reliably agree'} on which images are "
               f"faithfully explained (Spearman {fmt(a['value'], a['ci95_patient_bootstrap'])}).")
for line in verdict:
    print("-", line)

summary = {
    "lineage": {"checkpoint_md5": CKPT_MD5, "manifest_md5": MANIFEST_MD5, "sources": {k: list(v) for k, v in lineage.items()}},
    "n_test_images": int(N), "n_patients": int(len(np.unique(pids))), "n_with_faithfulness": n_faith,
    "settings": {"n_bootstrap": N_BOOT, "seed": SEED, "bootstrap_unit": "patient",
                 "confidence": "MC mean probability of the explained (deterministic) class",
                 "correctness": "correctness of the explained (deterministic) prediction",
                 "faith_score": "(ins - ins_random) - (del - del_random), per image; >0 = better than random map"},
    "metric_directions": {**{k: ("higher_better" if v > 0 else "lower_better") for k, v in TRUST.items()},
                          "mc_conf_explained": "higher_more_certain", "predictive_entropy": "higher_more_uncertain",
                          "mutual_information": "higher_more_uncertain"},
    "correlation_table": corr.to_dict("records"),
    "verdict": verdict,
    **SUMMARY,
}
with open(JOINT_DIR / "joint_summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=float)

back = pd.read_csv(JOINT_DIR / "joint_results.csv")
assert len(back) == len(METHODS) * N
assert (back.method.to_numpy() == np.tile(METHODS, N)).all() and (back.row.to_numpy() == np.repeat(np.arange(N), len(METHODS))).all()
for m in METHODS:
    assert (back.loc[back.method == m, "file_path"].to_numpy() == unc.file_path.to_numpy()).all()
assert back[["mc_conf_explained", "predictive_entropy", "iou"]].notna().any().all()
print(f"\njoint_results.csv: {len(back)} rows = {N} images x {len(METHODS)} explainers, order matches test_predictions.csv")
!ls -la /kaggle/working/results/joint /kaggle/working/results/figures

## The project's verdict: how to state it

The generated verdict above is the evidence. What follows is how to turn it into the claim in your
write-up, honestly, whichever way it came out.

**Can these explanations be trusted?** Answer per axis, never overall:
- **Faithfulness:** "better than a random map" is the bar (`faith_score` > 0, notebook 3's paired
  CIs). Clearing it means the heatmap carries *some* real information about what drives the
  prediction. It doesn't mean every pixel's highlighting is meaningful.
- **Stability:** high and tight after alignment means two near-identical scans get near-identical
  explanations. A long lower tail means that for some images they don't, and a clinician can't know
  in advance which images those are.
- **Localization:** pointing at the tumor shows *agreement with the radiologist*, not faithfulness.
  The two can come apart. A faithful map that's off-tumor reveals what the model really uses, which
  is a model problem, not an explainer problem.

**Is confidence a reliable proxy for explanation quality?** Only if the confidence–`faith_score`
correlation is positive *and* its CI excludes 0. Even then, check the magnitude: a Spearman of
0.15 is "detectable", not "reliable". A near-zero or negative result is the more common outcome in
the literature, and an important one: it means a high-confidence prediction says nothing about
whether its heatmap can be believed.

**Does uncertainty flag the bad cases?** Only the AUROCs whose CI lies above 0.5 count. If none do,
state it plainly: *MC-dropout uncertainty, at least in its last-layer form, does not identify
untrustworthy explanations*. And report the danger-cell count, since those images are the concrete
illustration (§12, category 2).

**Limitations to state alongside any claim:**
- **Last-layer MC-dropout** (notebook 4): one dropout layer before the head, so the backbone
  features are identical across passes. This is head uncertainty only. Full-network dropout, deep
  ensembles or test-time augmentation might relate to explanation quality differently.
- **One architecture** (ResNet18), **one dataset** (Figshare, 233 patients), **one split**: ~35 test
  patients and ~20 errors. Hence the patient-bootstrap CIs everywhere, and they're wide.
- **Sparse-map SSIM caveat** (notebook 3): shared empty background inflates Grad-CAM's SSIM, so the
  SSIM stability comparison favours Grad-CAM partly for structural reasons.
- **Deletion/insertion artefacts:** black-patch deletion is off-distribution, and IG's black baseline
  matches the deletion baseline, which slightly favours IG on deletion. `faith_score` controls for
  the per-image confidence confound, but not for these.
- **Per-image normalisation** (notebook 2) affects threshold-based localization (IoU/Dice at 0.5),
  not the rank-based or energy metrics.
- **Correlation, not causation:** the joint analysis shows associations across images. It doesn't
  show *why* confident images have better or worse explanations.

## Closing checklist

- [ ] Lineage check passed: all checkpoint and manifest MD5s identical.
- [ ] Merge: `2 × N` rows, nothing dropped, row order matches `test_predictions.csv` (asserted).
- [ ] Read the generated verdict, and check that the exemplar panels match it. If the verdict says
  "confidence predicts faithfulness" but the confident-faithful panels look no better than the
  confident-unfaithful ones, look again.
- [ ] Note `n_with_faithfulness`. If notebook 3 used `subset_n`, say so next to every faithfulness
  number.

**Save & Run All**
1. **Save Version → Save & Run All (Commit) → Save**, with all four inputs (notebooks 0, 2, 3, 4)
   attached. A GPU isn't needed; the accelerator can be *None*, which saves GPU quota.
2. Check the version's **Output** tab for `results/joint/` (`joint_results.csv`, `joint_summary.json`,
   `correlations.csv`) and the `joint_*.png` figures.
3. **Notebook 6 (README polish + demo)** attaches this output. `joint_summary.json` has every
   headline number with its CI, plus the generated verdict, so the README can quote numbers without
   recomputing anything.